# Named Entity Recognition with Pinecone

In this notebook, we explore Named Entity Recognition (NER) which is **automatically finding and classifying key pieces of information in text**, such as people, organizations, locations, or other specific entities.

NER is especially useful in search systems. When a database contains thousands of documents, simply relying on keyword search can return too many results. **By combining semantic similarity with entity extraction, we can narrow down the search space and surface more relevant results**.

You've already seen this idea in everyday life:
- Job boards that let you filter results by company or skill names
- News sites that let you browse by politician, city or sports team
- Research databases that let you filter by author or journal

In this notebook, we'll build a simplified version of that workflow. We will:
- **embed the query** - get semantic similarity scores with documents
- **extract named entities** - identify the most important keywords/topics
- **apply entity filters in Pinecone** - return results that are both semantically relevant and constrained to the specified entities

**In this notebook you will learn to:**
- Use a pretrained BERT-based NER model to extract named entities from text
- Combine NER with semantic search for more precise results
- Store entity metadata in Pinecone and use it for filtering
- Query a vector database using both embeddings and entity-based metadata filters

# Setup

Before we begin, let's install the necessary libraries, configure API access, and import all required packages.

**GPU Runtime:** This notebook runs a BERT NER model on ~2,500 articles and generates embeddings. For faster processing, switch to a GPU runtime:
1. Go to **Runtime** > **Change runtime type**
2. Select **T4 GPU** (or any available GPU)
3. Click **Save**

**Data File:** This notebook uses the `articles.csv` dataset. Upload it now while the libraries install:
1. Click the 📁 (Files) icon in the left sidebar
2. Click the upload button and select `articles.csv`
3. Wait for the upload to complete

## Install Dependencies

Run the cell below to install all dependencies for this notebook.

In [ ]:
!pip install -q sentence-transformers==5.3.0 transformers==5.0.0 pinecone==7.3.0

print("✅ All libraries installed successfully!")

## API Key Configuration

This notebook uses **Pinecone** as a managed vector database. You need a free Pinecone account:
1. Sign up at [pinecone.io](https://www.pinecone.io/)
2. Go to **API Keys** in the Pinecone console
3. Copy your API key

You have two methods to provide your API key:

**Method 1 (Recommended)**: Use Colab Secrets
1. Click the 🔑 icon in the left sidebar
2. Click "Add new secret"
3. Name: `PINECONE_API_KEY`
4. Value: Your Pinecone API key
5. Enable notebook access

**Method 2 (Fallback)**: Manual input when prompted

Run the cell below to configure authentication:

In [ ]:
import os

try:
    from google.colab import userdata
    PINECONE_API_KEY = userdata.get('PINECONE_API_KEY')
    print("✅ API key loaded from Colab secrets")
except:
    from getpass import getpass
    print("💡 To use Colab secrets: Go to 🔑 (left sidebar) → Add new secret → Name: PINECONE_API_KEY")
    PINECONE_API_KEY = getpass("Enter your Pinecone API Key: ")

os.environ["PINECONE_API_KEY"] = PINECONE_API_KEY

if not PINECONE_API_KEY or PINECONE_API_KEY.strip() == "":
    raise ValueError("❌ ERROR: No API key provided!")

print("✅ Pinecone authentication configured!")

## Import Libraries

In [ ]:
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForTokenClassification
from transformers import pipeline

import torch

import pandas as pd
import os
from pinecone import Pinecone, ServerlessSpec

import ast

from tqdm.auto import tqdm

# 1. Named Entity Recognition

We'll use a pre-trained BERT-based model called `dslim/bert-base-NER`. This model has been fine-tuned specifically for Named Entity Recognition tasks. It works at the token level - it looks at each word (or subword) in the input text and assigns it an entity label:
- `ORG`: organization
- `PER`: person's name
- `LOC`: location
- `MISC`: miscellaneous entity

You can read more about this model on [Hugging Face's website](https://huggingface.co/dslim/bert-base-NER).

Let's define the model name and load the tokenizer and model:

In [ ]:
model = "dslim/bert-base-NER"

Let's load the tokenizer which splits input text into tokens in exactly the same way the model was trained and prepares it for inference:

In [ ]:
# Loading the tokenizer
ner_tokenizer = AutoTokenizer.from_pretrained(model)

Now, we'll load the NER model:

In [ ]:
# Loading the NER model
ner_model = AutoModelForTokenClassification.from_pretrained(model)
ner_model

To run NER, we use a pipeline that wraps the model and tokenizer together and gives us a simple function we can call on text.

The pipeline also needs to know which **device** to use for computation. If a GPU is available (which it should be on Google Colab), the model will run on it - this is **significantly faster** than CPU.

In [ ]:
# Pipeline
# Use GPU if available (Google Colab provides one), otherwise CPU
device = 0 if torch.cuda.is_available() else -1

ner_pipeline = pipeline('ner',
                        # Model
                        model = ner_model,
                        # Tokenizer
                        tokenizer = ner_tokenizer,
                        # BERT splits words into subword pieces: option "max" groups those pieces back into whole words
                        # and keeps the label with the highest confidence for each word span
                        aggregation_strategy = "max",
                        # GPU / CPU
                        device = device)

print(f"NER pipeline running on: {'GPU' if device == 0 else 'CPU'}")

Our NER model identified and categorized several key named entities in the sentence. It recognized organizations like DeepMind, Meta, and Meta Superintelligence Labs as `ORG`, individuals like Demis Hassabis, Lex Fridman, and Mark Zuckerberg as `PER`, and AI as a miscellaneous entity (`MISC`). Each detection includes a high confidence score, along with the exact text span in which the entity appeared.

In [ ]:
ner_pipeline("""DeepMind CEO Demis Hassabis recently suggested on the Lex Fridman podcast that
Meta's aggressive hiring spree reflects its attempt to catch up in the AI race—
this includes launching Meta Superintelligence Labs under CEO Mark Zuckerberg.""")

### EXERCISE 1: Extract Entities from Your Own Text (5-7 minutes)

**What you'll practice:** Using the NER pipeline to identify named entities in text.

**Your task:**
1. Create a sentence or short paragraph with at least 3 named entities (people, organizations, or locations)
   - Example: "Microsoft CEO Satya Nadella announced new AI features in Seattle."
2. Run the NER pipeline on your text
3. Print the detected entities with their types (PER, ORG, LOC)
4. Analyze: Did the model correctly identify all entities? Were there any mistakes?

**Hint:** Use `ner_pipeline(your_text)` to extract entities. The result will show each entity with its type and position.

**Expected outcome:** You should see entities labeled as:
- PER (Person): Individual names
- ORG (Organization): Company/institution names
- LOC (Location): Place names

In [ ]:
# YOUR CODE HERE
# Example solution structure:
#
# my_text = """Your text with named entities here"""
#
# entities = ner_pipeline(my_text)
#
# print("Detected entities:")
# for entity in entities:
#     print(f"  {entity['word']}: {entity['entity']} (confidence: {entity['score']:.2f})")

# 2. Enhancing Semantic Search with NER

Now that we've seen how NER can automatically highlight people, places or organizations in text, let's put it to work in a real search scenario.

Imagine a tech article search platform. Users can type questions like "How to learn NLP?" or "Best Python libraries for deep learning". The system first finds articles semantically related to the question (so synonyms work), but it also uses Named Entity Recognition to find important terms (like "NLP", "Python", "PyTorch") and **filters results to only include articles that mention those entities**. This means if someone searches "NLP with PyTorch", they won't get generic NLP articles but **they'll get ones specifically mentioning PyTorch**.

That's exactly what we'll build in the upcoming demo. Here are the steps we'll follow:

1. Load and **preprocess the dataset** - we'll reuse the Medium dataset of 2,500 tech articles
2. **Extract named entities** from each article using a NER model
3. **Build metadata** that includes those entities along with other useful fields like title, authors and year
4. **Generate embeddings** for each article

Finally, we'll upsert everything into Pinecone. When we'll be querying the database, Pinecone will combine both to return results:
- **Vector similarity** (to capture semantic meaning)
- and **Entity metadata filters** (to narrow results to those mentioning specific terms)

## 2.1 Loading and Preprocessing the Dataset

The first step is loading the dataset and preprocessing of some of the columns - cleaning text, parsing lists (for columns "authors" and "tags"), creating new columns "timestamp_iso" and "year", removing missing value and creating unique identifiers for each article.

Let's load the dataset:

In [ ]:
# Loading
data = pd.read_csv("articles.csv")

Next, we define and apply preprocessing functions to clean the text, parse list columns, and create timestamp fields:

In [ ]:
def _safe_literal_eval(x):
    # Parsing a Python-literal string to a Python object
    if isinstance(x, str):
        try:
            return ast.literal_eval(x)
        except (ValueError, SyntaxError):
            return x
    return x

def preprocess_articles(df: pd.DataFrame) -> pd.DataFrame:
    dataset = data.copy()

    # "text" column: ensuring "text" is a stripped string
    if "text" in dataset.columns:
        dataset["text"] = dataset["text"].astype(str).str.strip()

    # Parsing columns "tags" and "authors" into Python lists
    if "tags" in dataset.columns:
        dataset["tags"] = dataset["tags"].apply(_safe_literal_eval)
    if "authors" in dataset.columns:
        def _to_list(x):
            if isinstance(x, list):
                return x
            if pd.notna(x):
                parsed = _safe_literal_eval(x)
                return parsed if isinstance(parsed, list) else [str(parsed)]
            return []
        dataset["authors"] = dataset["authors"].apply(_to_list)

    # Converting "timestamp" to pandas datetime objects. Creating new columns "timestamp_iso" and "year"
    if "timestamp" in dataset.columns:
        ts = pd.to_datetime(dataset["timestamp"], utc=True, format="mixed", errors="coerce")
        dataset["timestamp_iso"] = ts.dt.strftime("%Y-%m-%dT%H:%M:%SZ")
        dataset["year"] = ts.dt.year.astype("Int64")

    return dataset

Let's apply the preprocessing and inspect the result:

In [ ]:
# Calling preprocess_articles function
clean_data = preprocess_articles(data)

# Inspect the data
clean_data.head(2)

We'll also drop missing values and create a new column with unique IDs:

In [ ]:
# Dropping missing values
clean_data.dropna(inplace=True)

Let's create unique identifiers and inspect the result:

In [ ]:
# Creating unique identifiers
clean_data["id"] = clean_data.index.astype(str)

# Inspecting
clean_data.head(2)

## 2.2 Extracting Named Entities


The next step is extracting named entities. We need to make sure that each created entity list will be clean and free of duplicates. For example, if our NER model extracts ["Python", "Python", "NLP"], we don’t want duplicates cluttering our metadata. Thus, we will create a function called `dedupe_entities()` that removes duplicates:

In [ ]:
# Deduplicate extracted entities per document
def dedupe_entities(entity_lists):
    return [sorted(set(map(str, ents))) for ents in entity_lists]

Next, we'll create a helper function `extract_entities()`, which applies our previously initialized `ner_pipeline` (with the NER model from the first section of this notebook). This function takes a list of texts, runs the model on **the entire batch at once** and returns the named entities it detects. We'll later use it to extract entities for all articles in our dataset.

Two important details:
- **Truncation:** Each text is truncated to the first 1,000 characters before being passed to the model. The BERT-based NER model can only process up to 512 tokens (~1,000 characters), and key entities (people, organizations, topics) typically appear early in an article, so we don't lose much by trimming.
- **Batch processing:** We pass the entire batch at once using the pipeline's built-in `batch_size` parameter. This allows the model to process multiple texts in parallel, which is much faster (especially on a GPU).

In [ ]:
# Extracting named entities
# Truncate texts to first 1000 characters — BERT can only handle 512 tokens (~1000 chars),
# and key entities (people, orgs, topics) typically appear early in an article.
MAX_CHARS = 1000

def extract_entities(texts):
    truncated = [t[:MAX_CHARS] for t in texts]
    # Pass the whole batch to the pipeline at once
    batch_results = ner_pipeline(truncated, batch_size = len(truncated))
    return [[ent["word"] for ent in ents] for ents in batch_results]

Even with batch processing and truncation, running the NER model on thousands of articles can still use a lot of memory. So instead of processing the entire dataset at once, we split it into **smaller batches of 64 texts**. Each batch is processed in parallel by the pipeline, and the results are accumulated into one list:

In [ ]:
# Extracting entities in batches

BATCH_SIZE = 64
texts = clean_data["text"].astype(str).tolist()
all_entities = []

for i in tqdm(range(0, len(texts), BATCH_SIZE), desc = "Extracting entities"):
    batch = texts[i:i + BATCH_SIZE]
    # For each batch, run the function with the NER model
    batch_results = extract_entities(batch)
    # Accumulate results into one list
    all_entities.extend(batch_results)

# Deduplicate and add named entities as a new column
clean_data["named_entity"] = dedupe_entities(all_entities)

Let's inspect the extracted entities:

In [ ]:
# Inspect the data
clean_data.head()

## 2.3 Building Metadata

After extracting entities, we enrich our dataset with a new metadata column. This metadata combines several fields - title, authors, timestamp, year, tags - together with the extracted named entities. This ensures that when we query Pinecone later, we can filter not only by semantic similarity but also by these metadata attributes.

Let's create the metadata column and prepare the upsert dataframe:

In [ ]:
# Creating "metadata" column
clean_data['metadata'] = clean_data.apply(lambda x: {
    "title": x["title"],
    "authors": x["authors"],
    "timestamp": x["timestamp_iso"],
    "year": x["year"],
    "tags": x["tags"],
    "entities": x["named_entity"]
}, axis=1)

# Preparing dataframe for upsert
df_to_upsert = clean_data[["id", "metadata"]].copy()
df_to_upsert.head()

## 2.4 Generating Embeddings

Finally, we'll create embeddings for each article using the pre-trained `all-MiniLM-L6-v2` model from SentenceTransformers and store them in the dataframe as Python lists under a new column "values":

In [ ]:
# Loading embedding model
retriever = SentenceTransformer("all-MiniLM-L6-v2")

# Getting texts
texts = clean_data["text"].tolist()

# Creating embeddings
embeddings = retriever.encode(
    texts,
    batch_size=64,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=False
)

# Storing embeddings in DataFrame as lists
df_to_upsert["values"] = [vec.tolist() for vec in embeddings]

Let's preview the data ready for upserting:

In [ ]:
df_to_upsert.head()

## 2.5 Creating a Pinecone Index and Upserting the Data

Now that we’ve prepared our dataframe with unique IDs, metadata (including named entities) and vector embeddings, we’re ready to upsert everything into Pinecone.

Let's initialize the Pinecone client and create the index:

In [ ]:
pinecone_client = Pinecone(api_key=os.environ["PINECONE_API_KEY"])

Now let's create the index:

In [ ]:
# Creating an Index
pinecone_client.create_index(name = "articles-ner",
                             dimension = 384,
                             metric = "cosine",
                             spec = ServerlessSpec(
                                 cloud = "aws",
                                 region = "us-east-1"
                             ))

Let's connect to the index and upsert the data:

In [ ]:
# Connecting
index = pinecone_client.Index("articles-ner")

# Upserting the data
index.upsert_from_dataframe(
    df_to_upsert,
    batch_size = 500,
    show_progress = True)

## 2.6 Querying the Database

Now let's test our NER-enhanced search. We'll query for a topic and use extracted entities as filters.

First, we take our query and convert it to embedding:

In [ ]:
# Our question
query = "How to find work in UX?"

# Embedding question to query vector
emb_query = retriever.encode(query).tolist()

Next, we run our NER pipeline on the query to detect important terms. In this case, the model extracts the entity "UX", which we then use as a metadata filter:

In [ ]:
# Extracting named entity
entity_filter = extract_entities([query])[0]
entity_filter

Finally, we query Pinecone with both the embedding and the entity filter:

In [ ]:
responses = index.query(
    vector = emb_query,
    top_k= 5,
    include_metadata = True,
    # Using named entity for filtering
    filter = {"entities": {"$in" : entity_filter}})

Let's inspect the results:

In [ ]:
responses

The output shows the top 5 matches, each with a similarity score and the entities associated with the article.

### EXERCISE 2: Search with Entity Filtering (12-15 minutes)

**What you'll practice:** Combining semantic search with NER-based metadata filtering.

**Your task:**
1. Create a query that mentions a specific entity (person, organization, or location)
   - Examples: "What did Google announce?", "Articles about London", "News mentioning Elon Musk"
2. Extract entities from your query using the NER pipeline
3. Create an embedding for your query text
4. Query the Pinecone index with:
   - The query embedding (for semantic similarity)
   - An entity filter (to only return documents mentioning that entity)
5. Display the top 3 results with their scores and detected entities

**Hint:** The query structure is:
```python
index.query(
    vector=query_embedding,
    top_k=3,
    include_metadata=True,
    filter={"entities": {"$in": [entity_name]}}
)
```

**Expected outcome:** Results should be both semantically relevant AND contain the specific entity you filtered for. This is more precise than semantic search alone!

In [ ]:
# YOUR CODE HERE
# Example solution structure:
#
# # Step 1: Your query
# my_query = "Your query mentioning a specific entity"
#
# # Step 2: Extract entities
# query_entities = ner_pipeline(my_query)
# print("Query entities:", [e['word'] for e in query_entities])
#
# # Step 3: Create embedding
# query_embedding = embedding_model.encode(my_query, convert_to_numpy=True)
#
# # Step 4: Query with filter (if entities found)
# if query_entities:
#     entity_filter = query_entities[0]['word']  # Use first entity
#     results = index.query(
#         vector=query_embedding.tolist(),
#         top_k=3,
#         include_metadata=True,
#         filter={"entities": {"$in": [entity_filter]}}
#     )
#
#     print(f"\nResults filtered by entity '{entity_filter}':")
#     for i, match in enumerate(results['matches'], 1):
#         print(f"{i}. Score: {match['score']:.4f}")
#         print(f"   Entities: {match['metadata'].get('entities', [])}")
# else:
#     print("No entities found in query")